## Imports

In [ ]:
import os
import uuid
import asyncio
import dotenv
import openai
import pandas as pd
from datetime import datetime, timezone
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langsmith.schemas import RunTypeEnum
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
from ragas.metrics.collections import Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.dataset_schema import SingleTurnSample

dotenv.load_dotenv()

## Load Eval Dataset from LangSmith

In [ ]:
DATASET_NAME = "hephaestus-rag-eval"

ls_client = Client()
examples = list(ls_client.list_examples(dataset_name=DATASET_NAME))

sample_types = [e.metadata.get('sample_type') for e in examples]
print(f"Loaded {len(examples)} examples — single: {sample_types.count('single')}  multi: {sample_types.count('multi')}")

## RAG Pipeline

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "cm_interventions"


def embed_text(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def retrieve(query, top_k=5):
    vector = embed_text(query)
    hits = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=vector,
        limit=top_k,
    ).points
    return hits


def format_context(hits):
    parts = []
    for h in hits:
        p = h.payload
        parts.append(
            f"ID: {p.get('id')}\nMachine: {p.get('machine')}\n"
            f"Date: {p.get('date_start')}\nSummary: {p.get('summary')}"
        )
    return "\n\n---\n\n".join(parts)


RAG_PROMPT = """\
You are a maintenance assistant. Use the intervention records below to answer the question.
If the records do not contain relevant information, say \"I don't know\".
Be concise. Do not use markdown formatting.

Question: {question}

Records:
{context}
"""


def generate_answer(question, context):
    prompt = RAG_PROMPT.format(question=question, context=context)
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


def rag_pipeline(question, top_k=5):
    hits = retrieve(question, top_k)
    context = format_context(hits)
    answer = generate_answer(question, context)
    retrieved_ids = [h.payload.get('id') for h in hits]
    return answer, context, retrieved_ids

## Run Pipeline on Eval Set

In [ ]:
results = []

for ex in examples:
    question     = ex.inputs["question"]
    ground_truth = ex.outputs["ground_truth"]
    expected_ids = ex.outputs["chunks_id"]
    sample_type  = ex.metadata.get("sample_type")

    answer, context, retrieved_ids = rag_pipeline(question)

    results.append({
        "example_id":    str(ex.id),
        "sample_type":   sample_type,
        "question":      question,
        "answer":        answer,
        "ground_truth":  ground_truth,
        "context":       context,
        "retrieved_ids": retrieved_ids,
        "expected_ids":  expected_ids,
    })
    print(f"✓ [{sample_type}] {question[:70]}")

df_results = pd.DataFrame(results)
print(f"\nDone. {len(df_results)} rows.")

## RAGAS Evaluation

In [ ]:
# Wrap OpenAI models for RAGAS
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-nano", temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


async def ragas_context_precision_id(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_ids"],
        reference_context_ids=example["chunks_id"],
    )
    return await IDBasedContextPrecision().single_turn_ascore(sample)


async def ragas_context_recall_id(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_ids"],
        reference_context_ids=example["chunks_id"],
    )
    return await IDBasedContextRecall().single_turn_ascore(sample)


async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=[run["context"]],
    )
    return await Faithfulness(llm=ragas_llm).single_turn_ascore(sample)


async def ragas_answer_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=[run["context"]],
    )
    return await AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb).single_turn_ascore(sample)


METRIC_FNS = {
    "context_precision_id": ragas_context_precision_id,
    "context_recall_id":    ragas_context_recall_id,
    "faithfulness":         ragas_faithfulness,
    "answer_relevancy":     ragas_answer_relevancy,
}


async def score_row(run, example):
    scores = {}
    for name, fn in METRIC_FNS.items():
        try:
            scores[name] = await fn(run, example)
        except Exception as e:
            print(f"  ⚠ {name}: {e}")
            scores[name] = None
    return scores


async def run_eval(df):
    all_scores = []
    for _, row in df.iterrows():
        run     = row.to_dict()
        example = {"chunks_id": row["expected_ids"]}
        scores  = await score_row(run, example)
        all_scores.append(scores)
        print(f"✓ precision={scores['context_precision_id']:.2f}  "
              f"recall={scores['context_recall_id']:.2f}  "
              f"faithfulness={scores['faithfulness']:.2f}  "
              f"relevancy={scores['answer_relevancy']:.2f}")
    return pd.DataFrame(all_scores)


df_scores = await run_eval(df_results)
print(df_scores)

## Results Breakdown

In [ ]:
METRIC_COLS = list(METRIC_FNS.keys())

df_scores["sample_type"] = df_results["sample_type"].values

print("=== Overall ===")
print(df_scores[METRIC_COLS].mean().round(3).to_string())

print("\n=== By sample_type ===")
print(df_scores.groupby("sample_type")[METRIC_COLS].mean().round(3).to_string())

## Upload Results to LangSmith

In [ ]:
project_name = os.getenv("LANGSMITH_PROJECT", "hephaestus-agentic-maintenance")

for i, row in df_results.iterrows():
    scores = df_scores.iloc[i]
    run_id = uuid.uuid4()
    now    = datetime.now(timezone.utc)

    ls_client.create_run(
        id=run_id,
        name="rag-eval",
        run_type=RunTypeEnum.chain,
        project_name=project_name,
        inputs={"question": row["question"], "context": row["context"]},
        outputs={"answer": row["answer"]},
        reference_example_id=row["example_id"],
        start_time=now,
        end_time=now,
    )

    for metric in METRIC_COLS:
        score = scores[metric]
        if pd.notna(score):
            ls_client.create_feedback(
                run_id=run_id,
                key=metric,
                score=float(score),
            )

print(f"Uploaded {len(df_results)} runs with RAGAS scores to '{project_name}'")